In [1]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from pathlib import Path

In [18]:
# Output folders containing the generated images and metadata

output_dirs = ["sd35_output_behind_on_front", "sd35_output_below_over_under",
               "sd35_output_between_inside_outside", "sd35_output_holding_touching_not", 
               "sd35_output_left_right_above", "sd35_output_top", "sd35_output_not_without"]
# output_dirs = ["sd35_output_20251103_131309"]
# Class Pairs read from CSV file class_pairs_from_143851.csv
class_pairs = pd.read_csv("class_pairs_from_143851.csv", header=None).values
class_pairs = [(row[0], row[1]) for row in class_pairs]
# print(len(class_pairs))
# print(class_pairs)

# Phrases
phrases = ["above", "below", "under", "over", "to the left of", "to the right of", 
           "behind", "in front of", "on top of"]
problematic_phrases = ["on"]

In [19]:
# Function to load all metadata CSVs from the output directories
def load_all_metadata(dirs):
    all_metadata = []
    for dir_path in dirs:
        if not os.path.exists(dir_path):
            print(f"Warning: Directory {dir_path} not found, skipping...")
            continue
            
        for file in os.listdir(dir_path):
            if file.startswith("metadata_") and file.endswith(".csv"):
                file_path = os.path.join(dir_path, file)
                try:
                    df = pd.read_csv(file_path)
                    # Add the output directory to each row for later reference
                    df['output_dir'] = dir_path
                    all_metadata.append(df)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    
    if not all_metadata:
        raise ValueError("No metadata files found!")
        
    return pd.concat(all_metadata, ignore_index=True)


In [20]:
# Function to load tensor and get image by index
def get_image_from_tensor(output_dir, tensor_file, tensor_index):
    tensor_path = os.path.join(output_dir, "tensors", tensor_file)
    try:
        tensor_data = torch.load(tensor_path)
        img_tensor = tensor_data['images'][tensor_index]
        # Convert tensor to numpy array for display
        img_array = img_tensor.permute(1, 2, 0).numpy()
        # Clip values to valid range
        img_array = np.clip(img_array, 0, 1)
        return img_array
    except Exception as e:
        print(f"Error loading image from {tensor_path} at index {tensor_index}: {e}")
        return None


In [21]:
# Load all metadata
df = load_all_metadata(output_dirs)
print(f"Loaded metadata for {len(df)} images")

Loaded metadata for 1900 images


In [22]:
# 1. Visualize images for a specific phrase, one by one without interaction
def visualize_by_phrase(df, phrase_text):
    # Filter for the exact phrase
    phrase_df = df[df['phrase'] == phrase_text]
    
    if len(phrase_df) == 0:
        print(f"No images found for phrase: '{phrase_text}'")
        return
    
    print(f"\nPhrase: '{phrase_text}' ({len(phrase_df)} images found)")
    
    # Go through each image one by one
    for i, (_, row) in enumerate(phrase_df.iterrows()):
        plt.figure(figsize=(8, 6))
        
        img = get_image_from_tensor(row['output_dir'], row['tensor_file'], row['tensor_index'])
        print(row["tensor_file"])
        if img is not None:
            plt.imshow(img)
            plt.axis('off')
            
            # Add detailed title with all information
            plt.title(f"Image {i+1}/{len(phrase_df)}\n"
                     f"Phrase: '{row['phrase']}'\n"
                     f"Classes: {row['class1']} and {row['class2']}\n"
                     f"Prompt: '{row['prompt']}'", 
                     fontsize=10, wrap=True)
            
            plt.tight_layout()
            plt.show()
        else:
            print(f"Could not load image {i+1}/{len(phrase_df)}")


In [23]:
# 2. Visualize images for specific class1 and class2 combination, one by one without interaction
def visualize_by_classes(df, class1, class2):
    # Filter for the exact class1 and class2 in that order
    class_df = df[(df['class1'] == class1) & (df['class2'] == class2)]
    
    if len(class_df) == 0:
        print(f"No images found for class combination: '{class1}' and '{class2}'")
        return
    
    print(f"\nClass Combination: '{class1}' and '{class2}' ({len(class_df)} images found)")
    
    for i, (_, row) in enumerate(class_df.iterrows()):
        plt.figure(figsize=(8, 6))

        img = get_image_from_tensor(row['output_dir'], row['tensor_file'], row['tensor_index'])
        if img is not None:
            # Display image
            plt.imshow(img)
            plt.axis('off')
            
            # Add detailed title with all information
            plt.title(f"Image {i+1}/{len(class_df)}\n"
                     f"Classes: {row['class1']} and {row['class2']}\n"
                     f"Phrase: '{row['phrase']}'\n"
                     f"Prompt: '{row['prompt']}'", 
                     fontsize=10, wrap=True)
            
            plt.tight_layout()
            plt.show()
        else:
            print(f"Could not load image {i+1}/{len(class_df)}")

In [1]:
# # View all images for a specific phrase
# visualize_by_phrase(df, "to the left of")

"below" : 53
"above" : 53
left: 

In [10]:
# class1,class2 = class_pairs[1]

# # View all images for a specific class combination
# visualize_by_classes(df, "fork","snowboard")

In [24]:
# Visualize three image for a specific phrase
def visualize_three_by_phrase(df, phrase_text):
    # Filter for the exact phrase
    phrase_df = df[df['phrase'] == phrase_text]
    
    if len(phrase_df) == 0:
        print(f"No images found for phrase: '{phrase_text}'")
        return
    
    print(f"\nPhrase: '{phrase_text}' ({len(phrase_df)} images found)")
    count = 0
    for i, (_, row) in enumerate(phrase_df.iterrows()):
        plt.figure(figsize=(8, 6))
        
        img = get_image_from_tensor(row['output_dir'], row['tensor_file'], row['tensor_index'])
        print(row["tensor_file"])
        if img is not None:
            plt.imshow(img)
            plt.axis('off')
            
            # Add detailed title with all information
            plt.title(f"Image {i+1}/{len(phrase_df)}\n"
                     f"Phrase: '{row['phrase']}'\n"
                     f"Classes: {row['class1']} and {row['class2']}\n"
                     f"Prompt: '{row['prompt']}'", 
                     fontsize=10, wrap=True)
            
            plt.tight_layout()
            plt.show()
            count += 1
        else:
            print(f"Could not load image {i+1}/{len(phrase_df)}")
        if count > 2:
            break

In [29]:
# # View all images for a specific phrase
# visualize_three_by_phrase(df, "and not")